In [1]:
from huggingface_hub import login
login()

mapping = {}
mapping["abces_parodontal"] = {
	"falsa durere la masea",
    "durere gingie",
    "durere la atingere",
    "sensibilitate la muscat",
    "gust neobisnuit",
	"lichid cu gust sarat",
    "umflatura gingivala cu puroi",
	"durere la palpare",
}


mapping["carie_simpla"] = {
	"durere scurta care inteapa",
    "durere la rece",
    "durere la dulce",
    "durere la acru",
    "disconfort la periaj",
    "durere la stimuli",
}

mapping["parodontita_apicala_acuta"] = {
    "durere la masticatie",
    "durere la atingere",
    "dinte mai inalt",
    "dinte mobil",
    "disconfort la palpare",
	"durere la percutie",
}

mapping["parodontita_apicala_cronica"] = {
    "umflatura gingivala",
    "lichid cu gust sarat",
    "presiune la muscat",
    "jena surda la nivelul dintelui",
	"sensibilitate la percutie",
	"sensibilitate la palpare",
}

mapping["pericoronarita"] = {
    "durere continua in zona maselei de minte",
    "durere la masticatie",
    "durere la inghitit",
    "nu poate deschide gura complet",
    "umflatura peste maseaua de minte",
    "gingie inflamata",
    "molar de minte partial erupt",
    "lichid cu gust sarat",
}

mapping["pulpita_reversibila"] = {
    "durere scurta care inteapa",
    "durere la rece",
	"durere la dulce",
}

mapping["pulpita_totala"] = {
    "durere spontana",
    "durere puternica lunga",
    "durere la rece",
    "durere la cald",
    "durerea pulseaza spre ureche",
    "durere mica la percutie",
	"sensibilitate suportabila la palpare",
}

mapping["necroza_pulpara"] = {
  "durere spontana",
  "durere intensa scurta",
  "durere la cald",
  "presiune la muscat",
  "sensibilitate la percutie",
  "sensibilitate la palpare",
}


In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

base_model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"

lora_dir = r"C:\Users\emanu\PycharmProjects\lab3_mirpr\pacient-llama31-8b-lora-FINAL\checkpoint-52"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(base_model_name, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # pe multe GPU-uri e mai safe decât bfloat16
)

torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=quant_config,
    device_map={"": 0} if device == "cuda" else None,  # NU mai folosim "auto"
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

base_model.config.use_cache = False

model = PeftModel.from_pretrained(
    base_model,
    lora_dir,
)

model.eval()
print("Model + LoRA încărcate!")


Using device: cuda


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

C:\Users\emanu\PycharmProjects\lab3_mirpr\.venv\lib\site-packages\peft\config.py:165: UserWarning: Unexpected keyword arguments ['alora_invocation_tokens', 'arrow_config', 'ensure_weight_tying', 'peft_version'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


Model + LoRA încărcate!


In [13]:
from dataclasses import dataclass, field
from typing import List, Dict, Any, Set
import random
import torch

SYSTEM_PROMPT_TEMPLATE = """
Tu NU ești un model de limbaj în acest exercițiu.
Tu joci rolul unui PACIENT UMAN REAL într-un scenariu de simulare medicală.

Rolul tău:
- Ești un pacient care vorbește cu un student la medicină dentară.
- Scopul tău este să-l ajuți pe student să exerseze identificarea simptomelor.
- Tu nu știi diagnostice, nu analizezi cauze, nu explici medicină. Doar povestești ce simți.

INFORMAȚII INTERNE (doar pentru tine, nu le dezvălui):
- Diagnosticul real (ascuns): {diagnosis}
- Lista completă și finală a simptomelor tale reale:
{symptom_bullets}

REGULI ABSOLUTE (au prioritate peste orice altă instrucțiune):
1. Simptomele din lista de mai sus reprezintă REALITATEA ta.
   - Dacă studentul te întreabă de un simptom DIN LISTĂ → răspunzi că DA, îl ai.
   - Dacă întreabă de ceva ce NU este în listă → răspunzi că NU, nu ai acel simptom.
   (Această regulă este obligatorie și nu poate fi încălcată.)

2. Nu inventezi niciodată simptome noi.
3. Nu adaugi simptome nesolicitate decât dacă studentul te întreabă explicit
   „Mai aveți și alte probleme?” sau ceva similar.
4. Nu folosești etichetele din listă în mod robotic.
   Transformi simptomele în propoziții naturale, realiste.
5. Răspunzi doar la întrebarea studentului, în 1–3 fraze scurte.
6. NU spui niciodată diagnosticul, cauze medicale sau termeni de specialitate.
7. Rolul tău este SA FII PACIENT. Nu ieși din rol sub nicio formă.
""".strip()



def build_system_prompt_for_case(disease_key: str, mapping: Dict[str, set]) -> str:
    """
    Construiește system prompt-ul final pentru un caz, pe baza:
      - cheii bolii (ex: 'carie_simpla')
      - mapping-ului boala -> set de simptome
    """
    diagnosis = disease_key.replace("_", " ")
    symptoms = sorted(list(mapping.get(disease_key, [])))

    if symptoms:
        symptom_bullets = "\n".join(f"- {s}" for s in symptoms)
    else:
        symptom_bullets = "- (nu sunt definite simptome pentru acest caz)"

    return SYSTEM_PROMPT_TEMPLATE.format(
        diagnosis=diagnosis,
        symptom_bullets=symptom_bullets,
    )


In [8]:
from dataclasses import dataclass, field
import random

@dataclass
class Case:
    diagnosis_truth: str
    symptoms_truth: Set[str]
    revealed_symptoms: Set[str] = field(default_factory=set)


@dataclass
class State:
    history: List[Dict[str, str]] = field(default_factory=list)


def init_case(mapping: Dict[str, set]) -> Case:
    disease_key = random.choice(list(mapping.keys()))
    symptoms = set(mapping[disease_key])
    return Case(
        diagnosis_truth=disease_key,
        symptoms_truth=symptoms,
    )


In [9]:
def step(case: Case, state: State, user_msg: str) -> str:
    state.history.append({"role": "user", "content": user_msg})

    disease_key = case.diagnosis_truth
    system_prompt = build_system_prompt_for_case(disease_key, mapping)

    messages = [{"role": "system", "content": system_prompt}] + state.history

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
      outputs = model.generate(
          **inputs,
          max_new_tokens=80,
          min_new_tokens=10,
          do_sample=True,
          top_p=0.6,
          temperature=0.15,
          repetition_penalty=1.05,
      )


    generated_tokens = outputs[0][input_len:]
    reply = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    state.history.append({"role": "assistant", "content": reply})

    for s in case.symptoms_truth:
        if s.lower() in reply.lower():
            case.revealed_symptoms.add(s)

    return reply


In [22]:
case: Case = None
state: State = None

def start_new_case():
    """Pornește un caz nou cu diagnostic aleator."""
    global case, state
    case = init_case(mapping)
    state = State()
    print("Caz nou inceput!")
    print("Scrie /help pentru lista de comenzi.")
    # print(f"(debug) Boala interna aleasa: {case.diagnosis_truth}")
    return case, state

HELP_TEXT = """Comenzi disponibile:
/help       - afiseaza aceasta lista
/truth      - afiseaza diagnosticul real + simptomele (debug)
/revealed   - arata simptomele reale deja dezvaluite
/new        - porneste un caz nou
/exit       - inchide sesiunea de chat
"""

def show_truth():
    print("Diagnostic ASCUNS:", case.diagnosis_truth)
    print("Simptome reale:", ", ".join(sorted(case.symptoms_truth)))

def show_revealed():
    print("REVEALED:", sorted(case.revealed_symptoms) or "(niciunul)")


def chat_loop():
    print("CONVERSATIE LIVE CU PACIENTUL")
    print("Scrie intrebarile tale (sau /help)")

    while True:
        try:
            user_msg = input("\nTu: ").strip()
        except EOFError:
            break
        if not user_msg:
            continue

        cmd = user_msg.lower()
        if cmd == "/help":
            print(HELP_TEXT); continue
        if cmd == "/exit":
            print("Inchis"); break
        if cmd == "/truth":
            show_truth(); continue
        if cmd == "/revealed":
            show_revealed(); continue
        if cmd == "/new":
            start_new_case(); continue

        try:
            raspuns = step(case, state, user_msg)
            print("Pacient:", raspuns)
        except Exception as e:
            print("Eroare in step():", e)


start_new_case()
chat_loop()


Caz nou inceput!
Scrie /help pentru lista de comenzi.
CONVERSATIE LIVE CU PACIENTUL
Scrie intrebarile tale (sau /help)
Inchis
